# Modelado: Predicción de Argumentos Efectivos

CC3084 Data Science, Universidad del Valle de Guatemala, Semestre II 2026

Reto 19, Feedback Prize - Predicting Effective Arguments.

Este notebook continúa el análisis exploratorio. Aquí se entrenan y comparan los modelos.

## Decisiones que vienen del análisis exploratorio

**La partición se agrupa por ensayo.** Cada ensayo aporta 8.77 fragmentos en promedio que comparten autor, tema y nivel de escritura. Si se reparten al azar, el modelo vería fragmentos del mismo ensayo en entrenamiento y en prueba, y el resultado saldría inflado. Por eso se usa GroupShuffleSplit y GroupKFold con `essay_id` como grupo.

**La métrica es log loss.** Adequate concentra el 57.06 % de los casos, así que la exactitud no distingue un modelo útil de uno que siempre predice la clase mayoritaria. Se reporta también F1 macro, que trata a las tres clases por igual.

**Se incluyen las variables derivadas del texto.** Ninguna predice por sí sola, la mejor llegó a 0.289, pero combinadas pueden aportar. Se agregan también variables del ensayo completo, porque a ese nivel la correlación subió a 0.577.

In [ ]:
import os
import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import log_loss, f1_score, accuracy_score, classification_report, confusion_matrix

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Carga y construcción de variables

In [ ]:
ruta = '/kaggle/input/competitions/feedback-prize-effectiveness'
if not os.path.exists(ruta):
    ruta = 'data/raw'

datos = pd.read_csv(f'{ruta}/train.csv')
datos['discourse_text'] = datos['discourse_text'].str.strip()

print(datos.shape)
datos.head()

In [ ]:
def construir_variables(tabla):
    tabla = tabla.copy()
    tabla['n_caracteres'] = tabla['discourse_text'].str.len()
    tabla['n_palabras'] = tabla['discourse_text'].str.split().str.len()
    tabla['n_oraciones'] = tabla['discourse_text'].str.count(r'[.!?]').clip(lower=1)
    tabla['n_comas'] = tabla['discourse_text'].str.count(',')
    tabla['largo_palabra'] = tabla['n_caracteres'] / tabla['n_palabras']
    tabla['palabras_por_oracion'] = tabla['n_palabras'] / tabla['n_oraciones']
    tabla['log_palabras'] = np.log1p(tabla['n_palabras'])

    tabla['posicion'] = tabla.groupby('essay_id').cumcount()
    tabla['total_fragmentos'] = tabla.groupby('essay_id')['discourse_id'].transform('count')
    tabla['posicion_relativa'] = (tabla['posicion'] / (tabla['total_fragmentos'] - 1)).fillna(0.5)

    tabla['palabras_ensayo'] = tabla.groupby('essay_id')['n_palabras'].transform('sum')
    tabla['tipos_ensayo'] = tabla.groupby('essay_id')['discourse_type'].transform('nunique')
    tabla['proporcion_del_ensayo'] = tabla['n_palabras'] / tabla['palabras_ensayo']
    tabla['palabras_vs_tipo'] = tabla['n_palabras'] / tabla.groupby('discourse_type')['n_palabras'].transform('median')

    tabla['tipo_anterior'] = tabla.groupby('essay_id')['discourse_type'].shift(1).fillna('INICIO')
    tabla['tipo_siguiente'] = tabla.groupby('essay_id')['discourse_type'].shift(-1).fillna('FIN')
    return tabla


datos = construir_variables(datos)

numericas = ['n_caracteres', 'n_palabras', 'n_oraciones', 'n_comas', 'largo_palabra',
             'palabras_por_oracion', 'log_palabras', 'posicion', 'total_fragmentos',
             'posicion_relativa', 'palabras_ensayo', 'tipos_ensayo',
             'proporcion_del_ensayo', 'palabras_vs_tipo']

categoricas = ['discourse_type', 'tipo_anterior', 'tipo_siguiente']

datos[numericas].describe().T.round(2)

Las cuatro variables nuevas respecto al análisis exploratorio son `palabras_vs_tipo`, que compara la longitud contra la mediana de su propio tipo, `proporcion_del_ensayo`, que mide cuánto del ensayo ocupa el fragmento, y `tipo_anterior` junto con `tipo_siguiente`, que dan el contexto de los fragmentos vecinos.

Las tres salen de hallazgos del análisis exploratorio. La longitud solo se puede leer dentro de su tipo, y el contexto del ensayo pesa más que el fragmento aislado.

## 2. Partición agrupada por ensayo

In [ ]:
clases = ['Ineffective', 'Adequate', 'Effective']
objetivo = datos['discourse_effectiveness'].map({c: i for i, c in enumerate(clases)}).values
grupos = datos['essay_id'].values

separador = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
indice_entrena, indice_prueba = next(separador.split(datos, objetivo, grupos))

entrena = datos.iloc[indice_entrena].reset_index(drop=True)
prueba = datos.iloc[indice_prueba].reset_index(drop=True)
y_entrena = objetivo[indice_entrena]
y_prueba = objetivo[indice_prueba]

interno = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
indice_ajuste, indice_validacion = next(interno.split(entrena, y_entrena, entrena['essay_id']))

ajuste = entrena.iloc[indice_ajuste].reset_index(drop=True)
validacion = entrena.iloc[indice_validacion].reset_index(drop=True)
y_ajuste = y_entrena[indice_ajuste]
y_validacion = y_entrena[indice_validacion]

print('entrenamiento:', entrena.shape, 'ensayos:', entrena['essay_id'].nunique())
print('prueba:', prueba.shape, 'ensayos:', prueba['essay_id'].nunique())
print('ensayos compartidos:', len(set(entrena['essay_id']) & set(prueba['essay_id'])))
print('validación interna:', validacion.shape, 'ensayos:', validacion['essay_id'].nunique())

El conteo de ensayos compartidos debe dar cero. Esa es la comprobación de que la partición quedó bien agrupada.

La validación interna sale del propio conjunto de entrenamiento y también va agrupada por ensayo. Sirve para detener LightGBM en el momento justo. Usar el conjunto de prueba para eso sería hacer trampa, porque el modelo estaría eligiendo cuántos árboles construir mirando los datos con los que después se evalúa.

## 3. Línea base

Un clasificador que siempre predice la distribución de las clases. Sirve para saber contra qué comparar.

In [ ]:
resultados = {}


def evaluar(nombre, probabilidades, segundos):
    predicciones = probabilidades.argmax(axis=1)
    resultados[nombre] = {
        'log_loss': float(log_loss(y_prueba, probabilidades, labels=[0, 1, 2])),
        'f1_macro': float(f1_score(y_prueba, predicciones, average='macro')),
        'exactitud': float(accuracy_score(y_prueba, predicciones)),
        'segundos': float(segundos)
    }
    print(nombre)
    for clave, valor in resultados[nombre].items():
        print(f'  {clave}: {valor:.4f}')
    return predicciones

In [ ]:
vacio_entrena = np.zeros((len(entrena), 1))
vacio_prueba = np.zeros((len(prueba), 1))

inicio = time.time()
base = DummyClassifier(strategy='prior').fit(vacio_entrena, y_entrena)
evaluar('Línea base', base.predict_proba(vacio_prueba), time.time() - inicio)

## 4. Modelo 1. TF-IDF con regresión logística

Se concatena el tipo de argumento al inicio del texto para que el modelo lo vea como una señal más. Se usan n-gramas de palabras y de caracteres.

In [ ]:
entrena['texto_completo'] = entrena['discourse_type'] + ' [SEP] ' + entrena['discourse_text']
prueba['texto_completo'] = prueba['discourse_type'] + ' [SEP] ' + prueba['discourse_text']

vectorizador = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=60000,
    sublinear_tf=True,
    strip_accents='unicode'
)

inicio = time.time()
X_entrena = vectorizador.fit_transform(entrena['texto_completo'])
X_prueba = vectorizador.transform(prueba['texto_completo'])

regresion = LogisticRegression(C=2.0, max_iter=1500)
regresion.fit(X_entrena, y_entrena)

evaluar('TF-IDF + Regresión logística', regresion.predict_proba(X_prueba), time.time() - inicio)
print('vocabulario:', len(vectorizador.vocabulary_))

## 5. Modelo 2. LightGBM sobre variables derivadas

Aquí el texto entra reducido a 150 componentes con SVD, y se combinan con las variables numéricas y categóricas que construimos.

In [ ]:
inicio = time.time()

reductor = TruncatedSVD(n_components=150, random_state=42)
texto_entrena = reductor.fit_transform(X_entrena)
texto_prueba = reductor.transform(X_prueba)

codificador = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
cat_entrena = codificador.fit_transform(entrena[categoricas])
cat_prueba = codificador.transform(prueba[categoricas])

escalador = StandardScaler()
num_entrena = escalador.fit_transform(entrena[numericas])
num_prueba = escalador.transform(prueba[numericas])

validacion['texto_completo'] = validacion['discourse_type'] + ' [SEP] ' + validacion['discourse_text']
texto_validacion = reductor.transform(vectorizador.transform(validacion['texto_completo']))
cat_validacion = codificador.transform(validacion[categoricas])
num_validacion = escalador.transform(validacion[numericas])

tabla_entrena = np.hstack([num_entrena, cat_entrena, texto_entrena])
tabla_validacion = np.hstack([num_validacion, cat_validacion, texto_validacion])
tabla_prueba = np.hstack([num_prueba, cat_prueba, texto_prueba])

print('columnas del modelo:', tabla_entrena.shape[1])

In [ ]:
import lightgbm as lgb

bosque = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    n_estimators=1200,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=40,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1
)

bosque.fit(
    tabla_entrena, y_entrena,
    eval_set=[(tabla_validacion, y_validacion)],
    eval_metric='multi_logloss',
    callbacks=[lgb.early_stopping(80, verbose=False)]
)

evaluar('LightGBM', bosque.predict_proba(tabla_prueba), time.time() - inicio)
print('mejor iteración:', bosque.best_iteration_)

## 6. Validación cruzada agrupada

El resultado sobre una sola partición puede ser suerte. Con GroupKFold de 5 pliegues se ve qué tan estable es cada modelo.

In [ ]:
def validar(nombre, construir, n_pliegues=5):
    pliegues = GroupKFold(n_splits=n_pliegues)
    perdidas = []

    for numero, (idx_a, idx_b) in enumerate(pliegues.split(datos, objetivo, grupos), start=1):
        a = datos.iloc[idx_a]
        b = datos.iloc[idx_b]
        probabilidades = construir(a, objetivo[idx_a], b)
        perdida = log_loss(objetivo[idx_b], probabilidades, labels=[0, 1, 2])
        perdidas.append(perdida)
        print(f'  pliegue {numero}: {perdida:.4f}')

    print(f'{nombre}: {np.mean(perdidas):.4f} +/- {np.std(perdidas):.4f}')
    return perdidas


def construir_tfidf(a, y_a, b):
    texto_a = a['discourse_type'] + ' [SEP] ' + a['discourse_text']
    texto_b = b['discourse_type'] + ' [SEP] ' + b['discourse_text']
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=60000, sublinear_tf=True)
    modelo = LogisticRegression(C=2.0, max_iter=1500)
    modelo.fit(vec.fit_transform(texto_a), y_a)
    return modelo.predict_proba(vec.transform(texto_b))


perdidas_tfidf = validar('TF-IDF + Regresión logística', construir_tfidf)

## 7. Comparación de modelos

In [ ]:
comparacion = pd.DataFrame(resultados).T.round(4)
comparacion = comparacion.sort_values('log_loss')
comparacion

In [ ]:
figura, ejes = plt.subplots(1, 3, figsize=(16, 4.5))

for eje, metrica, titulo, mejor_bajo in zip(
    ejes,
    ['log_loss', 'f1_macro', 'segundos'],
    ['Log loss (menor es mejor)', 'F1 macro (mayor es mejor)', 'Tiempo de entrenamiento'],
    [True, False, True]
):
    orden = comparacion.sort_values(metrica, ascending=mejor_bajo)
    eje.barh(orden.index, orden[metrica], color='#4C72B0')
    eje.set_title(titulo)
    eje.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
mejor = comparacion.index[0]
probabilidades_mejor = {
    'TF-IDF + Regresión logística': lambda: regresion.predict_proba(X_prueba),
    'LightGBM': lambda: bosque.predict_proba(tabla_prueba)
}.get(mejor, lambda: base.predict_proba(prueba))()

predicciones = probabilidades_mejor.argmax(axis=1)
matriz = confusion_matrix(y_prueba, predicciones, normalize='true') * 100

plt.figure(figsize=(6.5, 5))
sns.heatmap(matriz.round(1), annot=True, fmt='.1f', cmap='Blues',
            xticklabels=clases, yticklabels=clases, cbar=False)
plt.title(f'Matriz de confusión, {mejor} (% por fila)')
plt.ylabel('real')
plt.xlabel('predicho')
plt.tight_layout()
plt.show()

print(classification_report(y_prueba, predicciones, target_names=clases, digits=3))

## 8. Guardado para la aplicación

In [ ]:
salida = '/kaggle/working' if os.path.exists('/kaggle/working') else 'modelos'
os.makedirs(salida, exist_ok=True)

joblib.dump(vectorizador, f'{salida}/vectorizador.joblib')
joblib.dump(regresion, f'{salida}/regresion.joblib')
joblib.dump({'reductor': reductor, 'codificador': codificador, 'escalador': escalador, 'modelo': bosque},
            f'{salida}/lightgbm.joblib')

with open(f'{salida}/metricas.json', 'w') as archivo:
    json.dump(resultados, archivo, indent=2)

referencias = {
    'mediana_por_tipo': datos.groupby('discourse_type')['n_palabras'].median().to_dict(),
    'mediana_ensayo': {
        'palabras_ensayo': float(datos['palabras_ensayo'].median()),
        'total_fragmentos': float(datos['total_fragmentos'].median()),
        'tipos_ensayo': float(datos['tipos_ensayo'].median()),
        'proporcion_del_ensayo': float(datos['proporcion_del_ensayo'].median()),
        'posicion': float(datos['posicion'].median()),
        'posicion_relativa': float(datos['posicion_relativa'].median())
    }
}

with open(f'{salida}/referencias.json', 'w') as archivo:
    json.dump(referencias, archivo, indent=2)

prueba[['discourse_id', 'essay_id', 'discourse_type', 'discourse_effectiveness'] + numericas].to_csv(
    f'{salida}/muestra_prueba.csv', index=False)

print(os.listdir(salida))

## 9. Pendiente. Modelo 3, DeBERTa

Este bloque necesita GPU y se corre aparte, porque tarda cerca de una hora. El código queda listo para ejecutarlo en Kaggle con el acelerador encendido.

La idea es la misma que ganó la competencia. Se le pasa al modelo el tipo de argumento junto con el texto del fragmento, para que aprenda la relación entre los dos.

In [ ]:
ENTRENAR_TRANSFORMER = False

if ENTRENAR_TRANSFORMER:
    import torch
    from torch.utils.data import Dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              TrainingArguments, Trainer)

    modelo_base = 'microsoft/deberta-v3-base'
    tokenizador = AutoTokenizer.from_pretrained(modelo_base)

    class Fragmentos(Dataset):
        def __init__(self, tabla, etiquetas):
            self.textos = (tabla['discourse_type'] + ' ' + tokenizador.sep_token + ' ' +
                           tabla['discourse_text']).tolist()
            self.etiquetas = etiquetas

        def __len__(self):
            return len(self.textos)

        def __getitem__(self, indice):
            codificado = tokenizador(self.textos[indice], truncation=True,
                                     max_length=512, padding='max_length')
            codificado = {k: torch.tensor(v) for k, v in codificado.items()}
            codificado['labels'] = torch.tensor(self.etiquetas[indice])
            return codificado

    def metricas(prediccion):
        logits, etiquetas = prediccion
        probabilidades = torch.softmax(torch.tensor(logits), dim=1).numpy()
        return {
            'log_loss': log_loss(etiquetas, probabilidades, labels=[0, 1, 2]),
            'f1_macro': f1_score(etiquetas, probabilidades.argmax(axis=1), average='macro')
        }

    transformer = AutoModelForSequenceClassification.from_pretrained(modelo_base, num_labels=3)

    configuracion = TrainingArguments(
        output_dir='deberta',
        num_train_epochs=2,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='log_loss',
        greater_is_better=False,
        fp16=True,
        report_to='none'
    )

    entrenador = Trainer(
        model=transformer,
        args=configuracion,
        train_dataset=Fragmentos(entrena, y_entrena),
        eval_dataset=Fragmentos(prueba, y_prueba),
        compute_metrics=metricas
    )

    entrenador.train()
    print(entrenador.evaluate())

## 10. Pendientes de esta etapa

**Correr el bloque de DeBERTa** con GPU y agregar su resultado a la tabla de comparación.

**Ajustar parámetros** de LightGBM con búsqueda aleatoria sobre `num_leaves`, `learning_rate` y `min_child_samples`. La guía pide documentar ese ajuste.

**Medir tiempos de inferencia** además de los de entrenamiento, porque la comparación entre algoritmos los pide.

**Conectar los artefactos guardados** con la aplicación de Streamlit que está en la carpeta `app`.